# Gradio Demo — public URL via ngrok

Launches the dual-mode CXR Intelligence app (Report + QA) and exposes it as a public URL.

**Prerequisites:** open this in the **same Kaggle session** that ran `kaggle_main.ipynb` (so the data, indices, and patched ColPali adapter are already on disk). Settings → Accelerator = GPU T4 ×2. Secrets: `HF_TOKEN`, `NVIDIA_API_KEY`, `NGROK_TOKEN` (free at https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
!pip install -q gradio pyngrok

In [ ]:
import os, sys, pathlib
sys.path.insert(0, '/kaggle/working/cxr-intel/src')
os.chdir('/kaggle/working/cxr-intel')

from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
os.environ['NVIDIA_API_KEY'] = s.get_secret('NVIDIA_API_KEY')
os.environ['LLM_PROVIDER'] = 'nvidia'
ngrok_token = s.get_secret('NGROK_TOKEN')

from cxr_intel.models.medgemma_runner import MedGemmaRunner
from cxr_intel.retrieval.colpali_index import ColPaliRetriever
from cxr_intel.generation.report_pipeline import ReportPipeline
from cxr_intel.generation.qa_pipeline import QAPipeline

medgemma = MedGemmaRunner(quantization='int4'); medgemma.load()
colpali = ColPaliRetriever(checkpoint='/kaggle/working/colpali-v1.3-patched', torch_dtype='float16')
colpali.load('data/indices/colpali_zs')

In [ ]:
import gradio as gr

def report(img):
    out = ReportPipeline('colpali_zs_rag', retriever=colpali, medgemma=medgemma, top_k=3).run(img)
    return out.report_text

def qa(img, q):
    out = QAPipeline('colpali_zs_rag', retriever=colpali, medgemma=medgemma, top_k=3).run(img, q)
    return out.answer

with gr.Blocks(title='CXR Intelligence — DSAI 413 A2') as demo:
    gr.Markdown('# CXR Intelligence — DSAI 413 A2')
    with gr.Tab('Report'):
        i = gr.Image(type='pil', label='Chest X-ray')
        o = gr.Textbox(label='Generated report', lines=8)
        gr.Button('Generate', variant='primary').click(report, [i], [o])
    with gr.Tab('QA'):
        i2 = gr.Image(type='pil', label='Chest X-ray')
        q2 = gr.Textbox(label='Question', placeholder='e.g. Is there cardiomegaly?')
        o2 = gr.Textbox(label='Answer', lines=4)
        gr.Button('Answer', variant='primary').click(qa, [i2, q2], [o2])

from pyngrok import ngrok
ngrok.set_auth_token(ngrok_token)
url = ngrok.connect(7860).public_url
print(f'\n\U0001f310 DEMO URL: {url}\n')
demo.launch(server_name='0.0.0.0', server_port=7860, quiet=True)